# 04 — Validate Model with Real Data

This notebook answers: **"Does the model actually work with my data?"**

We will:
1. Pick real countries from your dataset
2. Feed their numbers into the trained model
3. See what resources the model predicts
4. Test your own custom camp scenario

## 1. Load Models and Data

In [ ]:
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

MODELS_DIR = Path('../models')
PROCESSED  = Path('../data/processed')

# Load all 4 trained models
TARGETS = ['food_kg_day', 'water_liters_day', 'shelter_units', 'medical_staff_needed']
models  = {t: joblib.load(MODELS_DIR / f'resource_model_{t}.pkl') for t in TARGETS}

# Load the scaler — Ridge models need input scaled before predicting
scaler = joblib.load(MODELS_DIR / 'resource_scaler.pkl')

# Load metadata — tells us which model type was chosen per target
meta     = json.loads((MODELS_DIR / 'resource_models_meta.json').read_text())
FEATURES = meta['food_kg_day']['features']  # all 4 models use the same 21 features

# Track which targets were trained with Ridge (needs scaling) vs tree models (no scaling)
NEEDS_SCALING = {t: meta[t]['model_type'] == 'Ridge' for t in TARGETS}

# Load your processed camp data
df = pd.read_csv(PROCESSED / 'camp_features.csv')

print(f'Models loaded : {list(models.keys())}')
print(f'Features used : {len(FEATURES)}')
print(f'Dataset rows  : {len(df):,}')
print(f'Countries     : {df.country.nunique()}')
print(f'Year range    : {df.year.min()} - {df.year.max()}')
print()
print(f'{"Target":<30} {"Model":<22} {"Needs scaling?"}')
print('-' * 65)
for t in TARGETS:
    print(f'  {t:<28} {meta[t]["model_type"]:<22} {NEEDS_SCALING[t]}')

## 2. Helper — Build the Feature Vector

The model needs 21 features. This function takes a row from your data
and builds the full feature vector automatically.

In [ ]:
def build_features(row):
    """Turn one row of camp data into the 21-feature vector the model expects."""
    pop = row['total_population']
    d = {
        'year':                row.get('year', 2022),
        'refugees':            row.get('refugees', 0),
        'asylum_seekers':      row.get('asylum_seekers', 0),
        'returned_refugees':   row.get('returned_refugees', 0),
        'idps':                row.get('idps', 0),
        'returned_idps':       row.get('returned_idps', 0),
        'stateless':           row.get('stateless', 0),
        'ooc':                 row.get('ooc', 0),
        'total_population':    pop,
        'children_ratio':      row.get('children_ratio', 0.40),
        'elderly_ratio':       row.get('elderly_ratio', 0.05),
        'female_ratio':        row.get('female_ratio', 0.49),
        'yearly_arrivals':     row.get('yearly_arrivals', 0),
        'arrival_growth_rate': row.get('arrival_growth_rate', 0),
        'log_refugees':        np.log1p(row.get('refugees', 0)),
        'log_asylum_seekers':  np.log1p(row.get('asylum_seekers', 0)),
        'log_idps':            np.log1p(row.get('idps', 0)),
        'log_total_population':np.log1p(pop),
        'log_yearly_arrivals': np.log1p(row.get('yearly_arrivals', 0)),
        'years_elapsed':       row.get('year', 2022) - 2010,
        'vulnerability_index': row.get('children_ratio', 0.40) + row.get('elderly_ratio', 0.05),
    }
    return [d[f] for f in FEATURES]


def predict_resources(row):
    """Run all 4 models and return a dict of resource predictions.

    Ridge models were trained on scaled data, so we apply the scaler before
    predicting with them. Tree-based models are scale-invariant and get raw features.
    """
    raw_features = build_features(row)
    X_raw    = pd.DataFrame([raw_features], columns=FEATURES)
    X_scaled = scaler.transform(X_raw)

    results = {}
    for t in TARGETS:
        X_input = X_scaled if NEEDS_SCALING[t] else X_raw
        results[t] = round(float(models[t].predict(X_input)[0]), 1)

    results['shelter_units'] = int(results['shelter_units'])
    return results


print('Helper functions ready.')
print()
# Sanity check on one real row
sample = df[df['total_population'] > 1_000_000].iloc[0]
pred   = predict_resources(sample)
print(f'Sanity check — {sample["country"]} {int(sample["year"])}:')
print(f'  Population  : {int(sample["total_population"]):>12,}')
print(f'  Food (kg/d) : {pred["food_kg_day"]:>12,.1f}   actual: {sample["food_kg_day"]:,.1f}')
print(f'  Shelter     : {pred["shelter_units"]:>12,}   actual: {sample["shelter_units"]:,}')
print(f'  Medical     : {pred["medical_staff_needed"]:>12,.1f}   actual: {sample["medical_staff_needed"]:,.1f}')

## 3. Test on Real Countries from Your Dataset

Pick any countries you want to check. We compare:
- **Actual** = the UNHCR standard value we calculated from population
- **Predicted** = what the model outputs when it only sees the features (not the target)

In [ ]:
# ── Change these countries to any in your dataset ─────────────────────────
TEST_COUNTRIES = ['Syrian Arab Rep.', 'Dem. Rep. of the Congo', 'Colombia',
                  'Afghanistan', 'Ethiopia']
TEST_YEAR      = 2020
# ──────────────────────────────────────────────────────────────────────────

rows = df[(df['country'].isin(TEST_COUNTRIES)) & (df['year'] == TEST_YEAR)]

if rows.empty:
    print(f'No data found for year {TEST_YEAR}. Available years: {sorted(df.year.unique())}')
else:
    records = []
    for _, row in rows.iterrows():
        pred = predict_resources(row)
        records.append({
            'Country':          row['country'],
            'Population':       f"{int(row['total_population']):,}",
            'Children %':       f"{row['children_ratio']*100:.1f}%",
            '--- FOOD (kg/day) ---': '',
            'Actual food':      f"{row['food_kg_day']:,.0f}",
            'Predicted food':   f"{pred['food_kg_day']:,.0f}",
            '--- SHELTER (units) ---': '',
            'Actual shelter':   f"{row['shelter_units']:,}",
            'Predicted shelter':f"{pred['shelter_units']:,}",
            '--- MEDICAL (staff) ---': '',
            'Actual medical':   f"{row['medical_staff_needed']:,.0f}",
            'Predicted medical':f"{pred['medical_staff_needed']:,.0f}",
        })

    results_df = pd.DataFrame(records)
    print(f'Results for year {TEST_YEAR}:')
    print()
    display(results_df)

## 4. Visual Comparison — Predicted vs Actual

In [ ]:
rows = df[(df['country'].isin(TEST_COUNTRIES)) & (df['year'] == TEST_YEAR)].copy()

if not rows.empty:
    actuals    = {t: [] for t in TARGETS}
    predictions= {t: [] for t in TARGETS}
    countries  = []

    for _, row in rows.iterrows():
        pred = predict_resources(row)
        countries.append(row['country'].replace('Dem. Rep. of the Congo', 'DRC')
                                       .replace('Syrian Arab Rep.', 'Syria'))
        for t in TARGETS:
            actuals[t].append(row[t])
            predictions[t].append(pred[t])

    labels = ['Food\n(kg/day)', 'Water\n(L/day)', 'Shelter\n(units)', 'Medical\n(staff)']
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    for ax, target, label in zip(axes.flat, TARGETS, labels):
        x     = np.arange(len(countries))
        width = 0.35
        ax.bar(x - width/2, actuals[target],     width, label='Actual (UNHCR std)', color='#1565C0', alpha=0.85)
        ax.bar(x + width/2, predictions[target], width, label='Model Prediction',   color='#E53935', alpha=0.85)
        ax.set_xticks(x)
        ax.set_xticklabels(countries, rotation=20, ha='right', fontsize=9)
        ax.set_title(label, fontsize=12, fontweight='bold')
        ax.legend(fontsize=9)
        ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v/1e6:.1f}M' if v >= 1e6
                                                                     else f'{v/1e3:.0f}K' if v >= 1e3
                                                                     else f'{v:.0f}'))
    plt.suptitle(f'Model Predictions vs UNHCR Standard — {TEST_YEAR}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 5. Track One Country Over Time

See how resource needs changed year by year — and whether the model tracks the real trend.

In [ ]:
# ── Change this to any country in your dataset ───────────────────────────
COUNTRY = 'Syrian Arab Rep.'
# ─────────────────────────────────────────────────────────────────────────

country_df = df[df['country'] == COUNTRY].sort_values('year')

if country_df.empty:
    print(f'Country "{COUNTRY}" not found.')
    print('Available countries (sample):', sorted(df.country.unique())[:20])
else:
    years       = country_df['year'].tolist()
    actual_food = country_df['food_kg_day'].tolist()
    pred_food   = [predict_resources(row)['food_kg_day'] for _, row in country_df.iterrows()]
    actual_med  = country_df['medical_staff_needed'].tolist()
    pred_med    = [predict_resources(row)['medical_staff_needed'] for _, row in country_df.iterrows()]
    population  = country_df['total_population'].tolist()

    fig, axes = plt.subplots(3, 1, figsize=(13, 11), sharex=True)

    # Population
    axes[0].fill_between(years, [p/1e6 for p in population], alpha=0.3, color='gray')
    axes[0].plot(years, [p/1e6 for p in population], 'o-', color='gray', linewidth=2)
    axes[0].set_ylabel('Population (millions)')
    axes[0].set_title(f'{COUNTRY} — Population & Resource Needs Over Time', fontsize=13)
    axes[0].grid(True, alpha=0.3)

    # Food
    axes[1].plot(years, [f/1e3 for f in actual_food], 'o-', color='#1565C0', linewidth=2, label='Actual (UNHCR standard)')
    axes[1].plot(years, [f/1e3 for f in pred_food],   's--', color='#E53935', linewidth=2, label='Model Prediction')
    axes[1].set_ylabel('Food needed (tonnes/day)')
    axes[1].legend(fontsize=9)
    axes[1].grid(True, alpha=0.3)

    # Medical
    axes[2].plot(years, actual_med, 'o-', color='#2E7D32', linewidth=2, label='Actual (UNHCR standard)')
    axes[2].plot(years, pred_med,   's--', color='#E53935', linewidth=2, label='Model Prediction')
    axes[2].set_ylabel('Medical staff needed')
    axes[2].set_xlabel('Year')
    axes[2].legend(fontsize=9)
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print(f'\nData for {COUNTRY}:')
    compare = pd.DataFrame({
        'Year': years,
        'Population': [f'{p:,.0f}' for p in population],
        'Food actual (kg)': [f'{f:,.0f}' for f in actual_food],
        'Food predicted (kg)': [f'{p:,.0f}' for p in pred_food],
        'Medical actual': [f'{m:,.0f}' for m in actual_med],
        'Medical predicted': [f'{p:,.0f}' for p in pred_med],
    })
    display(compare)

## 6. Test Your Own Camp Scenario

Type in any numbers you want — the model will tell you what resources that camp needs.  
This is the real-world use case: a new camp opens, you count heads, model gives you the plan.

In [ ]:
# ── EDIT THESE NUMBERS to describe your camp scenario ────────────────────
MY_CAMP = {
    'year':            2024,
    'refugees':        8000,       # people who fled another country
    'idps':            2000,       # internally displaced (still in same country)
    'asylum_seekers':  500,        # people waiting for refugee status
    'stateless':       0,
    'ooc':             0,
    'returned_refugees': 0,
    'returned_idps':   0,
    'total_population': 10500,     # total everyone in the camp

    'children_ratio':  0.45,       # 45% are children under 18
    'elderly_ratio':   0.08,       # 8% are elderly 60+
    'female_ratio':    0.51,       # 51% female

    'yearly_arrivals':    1200,    # people arrived this year
    'arrival_growth_rate': 0.15,   # growing 15% compared to last year
}
# ─────────────────────────────────────────────────────────────────────────

pred = predict_resources(MY_CAMP)
pop  = MY_CAMP['total_population']

print('=' * 55)
print(f'  RESOURCE PLAN FOR YOUR CAMP ({pop:,} people)')
print('=' * 55)
print()
print(f'  Children (<18)  : {MY_CAMP["children_ratio"]*100:.0f}%  ({pop*MY_CAMP["children_ratio']:,.0f} people)')
print(f'  Elderly (60+)   : {MY_CAMP["elderly_ratio"]*100:.0f}%  ({pop*MY_CAMP["elderly_ratio"]:,.0f} people)')
print(f'  Female          : {MY_CAMP["female_ratio"]*100:.0f}%  ({pop*MY_CAMP["female_ratio"]:,.0f} people)')
print()
print(f'  Food needed     : {pred["food_kg_day"]:>10,.1f}  kg/day')
print(f'  Water needed    : {pred["water_liters_day"]:>10,.1f}  litres/day')
print(f'  Shelter units   : {pred["shelter_units"]:>10,}  units')
print(f'  Medical staff   : {pred["medical_staff_needed"]:>10,.1f}  workers')
print()
print(f'  Per person/day  :')
print(f'    Food   : {pred["food_kg_day"]/pop:.3f} kg')
print(f'    Water  : {pred["water_liters_day"]/pop:.1f} litres')
print(f'    Shelter: 1 unit per {pop/pred["shelter_units"]:.1f} people')
print('=' * 55)

## 7. Explore All Countries in Your Data

See every country available and find one you want to study.

In [ ]:
# Top 20 countries by total displaced population (all years combined)
top_countries = (
    df.groupby('country')['total_population']
    .sum()
    .sort_values(ascending=False)
    .head(20)
)

fig, ax = plt.subplots(figsize=(12, 7))
top_countries.div(1e6).sort_values().plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Top 20 Countries by Total Displaced Population (all years)', fontsize=13)
ax.set_xlabel('Total People (millions, summed across all years)')
plt.tight_layout()
plt.show()

print('\nAll countries in your dataset:')
all_countries = sorted(df.country.unique())
for i, c in enumerate(all_countries):
    print(f'  {c}', end='\n' if (i+1) % 3 == 0 else '\t')